# The Complete Customer Churn Classifier Guide

Welcome to the complete, step-by-step notebook for building a **Customer Churn Classifier**! 

Customer churn occurs when customers stop doing business with a company. For a telecom company, this means canceling their subscription. Predicting which customers are at risk of leaving allows a company to proactively offer discounts, better plans, or incentives to keep them.

We will go through this project continuously in **5 core phases**:
1. **Phase 1: Exploratory Data Analysis (EDA)** — Visualizing customer features and finding patterns.
2. **Phase 2: Data Preprocessing & Feature Engineering** — Cleaning missing values, converting text columns to numbers, scaling, and splitting the data.
3. **Phase 3: Model Training** — Training baseline and advanced models (Logistic Regression, Random Forest, XGBoost).
4. **Phase 4: Model Evaluation & Handling Class Imbalance** — Calculating Precision, Recall, Confusion Matrices, and adjusting class weights to catch more churners.
5. **Phase 5: Feature Importance & Business Insights** — Inspecting what factors drive churn and creating actionable business recommendations.

--- 
# Phase 1: Exploratory Data Analysis (EDA)

Let's import our libraries, load the Telco Customer Churn dataset directly from a public URL, and examine what the data looks like.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set styling for our charts
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Load dataset
dataset_url = "https://raw.githubusercontent.com/treselle-systems/customer_churn_analysis/master/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(dataset_url)

print(f"Dataset Shape: {df.shape[0]} rows (customers), {df.shape[1]} columns (features)")
df.head()

### Column Information and Structure
Let's check the data types and see if there are missing values.

In [ ]:
df.info()

### Observations:
- `customerID` is a unique text key. We will drop this since it doesn't help make generalizations.
- `Churn` is our target column (Yes/No).
- `TotalCharges` is listed as `object` (text) rather than a number. This means there might be empty spaces in it that prevent Pandas from reading it as a float. Let's verify this.

In [ ]:
# Find rows with empty spaces in TotalCharges
blank_charges = df[df['TotalCharges'].str.strip() == '']
print(f"Number of blank spaces in TotalCharges: {len(blank_charges)}")
blank_charges[['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']].head()

Notice that for these 11 customers, their `tenure` is `0` (months with the company). Since they are brand new customers, they haven't been billed yet! We will clean this in Phase 2.

### Visualizing Churn (Class Imbalance)
Let's see how many customers stayed (`No`) versus left (`Yes`).

In [ ]:
churn_counts = df['Churn'].value_counts()
churn_rate = (churn_counts['Yes'] / len(df)) * 100
print(f"Churn counts:\n{churn_counts}\n")
print(f"Churn rate: {churn_rate:.2f}%")

plt.figure(figsize=(6, 5))
sns.countplot(x='Churn', data=df, palette='Set2')
plt.title('Distribution of Customer Churn')
plt.xlabel('Churn Status')
plt.ylabel('Number of Customers')
plt.show()

Around 26.5% of the customers churned. This is an **imbalanced dataset** (more non-churners than churners). We need to remember this when evaluating model metrics later.

### Analyzing Key Attributes vs. Churn
Let's see how Contract type and Internet Service correlate with Churn.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Contract Type
sns.countplot(x='Contract', hue='Churn', data=df, palette='Set2', ax=axes[0])
axes[0].set_title('Churn by Contract Type')
axes[0].set_xlabel('Contract Type')
axes[0].set_ylabel('Count')

# Internet Service
sns.countplot(x='InternetService', hue='Churn', data=df, palette='Set2', ax=axes[1])
axes[1].set_title('Churn by Internet Service Type')
axes[1].set_xlabel('Internet Service')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

- **Month-to-month contracts** have a much higher churn rate than 1 or 2-year contracts.
- **Fiber optic internet service** customers have a surprisingly high rate of churn compared to DSL.

### Numerical Attributes (Tenure & Monthly Charges)
Let's inspect how the duration of customer relationships (`tenure`) and costs (`MonthlyCharges`) correlate with churn.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Tenure distribution
sns.kdeplot(data=df, x='tenure', hue='Churn', fill=True, common_norm=False, palette='Set2', alpha=0.5, ax=axes[0])
axes[0].set_title('Customer Tenure Distribution')
axes[0].set_xlabel('Tenure (Months)')
axes[0].set_ylabel('Density')

# Monthly Charges distribution
sns.kdeplot(data=df, x='MonthlyCharges', hue='Churn', fill=True, common_norm=False, palette='Set2', alpha=0.5, ax=axes[1])
axes[1].set_title('Monthly Charges Distribution')
axes[1].set_xlabel('Monthly Charges ($)')
axes[1].set_ylabel('Density')

plt.tight_layout()
plt.show()

- **Tenure**: Customers are highly prone to churning in their first year (tenure < 12). Long-tenured customers (> 60 months) rarely churn.
- **Monthly Charges**: Customers who churn peak around $70–$100, while loyal customers are more common around the cheaper $20 range.

--- 
# Phase 2: Data Preprocessing & Feature Engineering

Now we clean the data, convert text columns into numerical categories, scale numerical variables, and split the data into training and testing sets.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Drop redundant columns
df_clean = df.drop(columns=['customerID'])

# 2. Clean TotalCharges (convert to numeric, replace spaces with NaN, fill NaN with 0.0)
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce').fillna(0.0)

# 3. Map binary categorical columns to 1 and 0
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'MultipleLines', 
               'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
               'TechSupport', 'StreamingTV', 'StreamingMovies', 
               'PaperlessBilling', 'Churn']
for col in binary_cols:
    df_clean[col] = df_clean[col].apply(lambda x: 1 if x == 'Yes' else 0)

# Map gender (Female = 1, Male = 0)
df_clean['gender'] = df_clean['gender'].apply(lambda x: 1 if x == 'Female' else 0)

# 4. One-Hot Encode multi-class categories (Contract, InternetService, PaymentMethod)
multi_cat_cols = ['InternetService', 'Contract', 'PaymentMethod']
df_clean = pd.get_dummies(df_clean, columns=multi_cat_cols, drop_first=True)

# Convert boolean dummies to 0/1 integers
bool_cols = df_clean.select_dtypes(include=['bool']).columns
df_clean[bool_cols] = df_clean[bool_cols].astype(int)

print(f"Processed dataset dimensions: {df_clean.shape}")
df_clean.head(2)

### Train-Test Split & Scaling
We separate the labels, split the data (80% train, 20% test) with stratification, and scale the numeric features.

In [ ]:
# Separate features and target
X = df_clean.drop(columns=['Churn'])
y = df_clean['Churn']

# Split the data with stratification to maintain class proportions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale numerical features (tenure, MonthlyCharges, TotalCharges)
scaler = StandardScaler()
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Fit ONLY on the training data, then transform both sets
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

print(f"Train size: {X_train.shape[0]} rows, Test size: {X_test.shape[0]} rows")
X_train_scaled[num_cols].head()

--- 
# Phase 3: Model Training

We will train three different types of classifiers:
1. **Logistic Regression** (Linear baseline)
2. **Random Forest** (Tree-based ensemble)
3. **XGBoost** (Gradient boosted sequential trees)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# 1. Initialize models
log_reg = LogisticRegression(random_state=42, max_iter=1000)
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
xgb_clf = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42, eval_metric='logloss')

# 2. Train models
log_reg.fit(X_train_scaled, y_train)
rf_clf.fit(X_train_scaled, y_train)
xgb_clf.fit(X_train_scaled, y_train)

# 3. Print out basic accuracies
print(f"Logistic Regression Test Accuracy: {log_reg.score(X_test_scaled, y_test):.4f}")
print(f"Random Forest Test Accuracy:       {rf_clf.score(X_test_scaled, y_test):.4f}")
print(f"XGBoost Test Accuracy:             {xgb_clf.score(X_test_scaled, y_test):.4f}")

--- 
# Phase 4: Model Evaluation & Handling Class Imbalance

All three models score ~80% accuracy. Let's inspect detailed evaluation metrics. We will calculate:
- **Precision**: Correctly predicted churn / total predicted churn.
- **Recall**: Correctly predicted churn / total actual churn.
- **F1-Score**: Summary harmonic mean.
- **ROC-AUC Score**: Ability to separate the classes.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score

models = {
    'Logistic Regression': log_reg,
    'Random Forest': rf_clf,
    'XGBoost': xgb_clf
}

for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    auc_score = roc_auc_score(y_test, y_prob)
    
    print(f"==================== {name} ====================")
    print(f"ROC-AUC: {auc_score:.4f}")
    print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))
    print()

Notice that the **Recall for Churn** is around **0.50–0.55**. We are failing to predict almost half of the customers who are going to churn!

### Visualizing Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (name, model) in enumerate(models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[idx],
                xticklabels=['Pred Stay', 'Pred Churn'],
                yticklabels=['Actual Stay', 'Actual Churn'])
    axes[idx].set_title(f'{name} Confusion Matrix')
    
plt.tight_layout()
plt.show()

### Plotting ROC Curves

In [ ]:
plt.figure(figsize=(8, 6))

for name, model in models.items():
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_score = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_score:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Guess (AUC = 0.500)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curves')
plt.legend(loc="lower right")
plt.show()

### Improving Recall: Setting Class Weights
We can fix the bias by using `class_weight='balanced'`. This penalizes incorrect churn predictions more heavily during training, forcing models to catch more churners.

In [ ]:
# Train balanced classifiers
log_reg_balanced = LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
rf_balanced = RandomForestClassifier(class_weight='balanced', max_depth=8, random_state=42)

log_reg_balanced.fit(X_train_scaled, y_train)
rf_balanced.fit(X_train_scaled, y_train)

balanced_models = {
    'Balanced Logistic Regression': log_reg_balanced,
    'Balanced Random Forest': rf_balanced
}

for name, model in balanced_models.items():
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    auc_val = roc_auc_score(y_test, y_prob)
    
    print(f"==================== {name} ====================")
    print(f"ROC-AUC: {auc_val:.4f}")
    print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))
    print()

Excellent! The **Recall for Churn** went from **~52%** to **~80%**! Even though Precision goes down slightly (meaning some false alarms), catching 80% of our churners is a massive upgrade for a business retention team.

--- 
# Phase 5: Feature Importance & Business Insights

Let's extract the feature importances from our Balanced Random Forest model to see which factors drive customer decisions.

In [ ]:
# Get importances
importances = rf_balanced.feature_importances_
feature_names_list = list(X.columns)

# Sort importances
feat_imp_df = pd.DataFrame({
    'Feature': feature_names_list,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Plot the Top 15 Features
sns.barplot(x='Importance', y='Feature', data=feat_imp_df.head(15), palette='viridis')
plt.title('Top 15 Drivers of Customer Churn')
plt.xlabel('Feature Importance Score')
plt.ylabel('Feature Name')
plt.show()

### Translation to Business Actions

Based on the results above, we can propose four concrete business initiatives:

1. **Contract Management**: Month-to-month contracts are highly predictive of churn. Offer discounts or rewards to month-to-month customers who upgrade to 1 or 2-year agreements.
2. **Onboarding Support**: Customer tenure is a dominant driver. Establish a dedicated onboarding program for new customers (tenure < 12 months) to help them set up their services easily and stay active.
3. **Evaluate Fiber Optic Pricing/Quality**: High monthly charges and fiber optic subscriptions correlate with churn. Audit the service quality of fiber optic nodes and offer service adjustments or loyalty discounts for high-paying subscribers.
4. **Automated CRM Flagging**: Program your customer database to flag customers who have a high churn probability (calculated using `model.predict_proba(X)`) and route their queries directly to a dedicated retention unit.